## Chargement des datasets et utilitaires de base

In [1]:
# === Cellule 1 : imports et données de base ===

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

import trimesh

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# À adapter à ton projet : ici j'imagine un settings.PROCESSED_DATA_PATH
from settings import PROCESSED_DATA_PATH

# Chargement des dataframes préparés dans exploration.ipynb
df_lower = pd.read_pickle(PROCESSED_DATA_PATH / "lower_jaw_landmark_dataset.pkl")
df_upper = pd.read_pickle(PROCESSED_DATA_PATH / "upper_jaw_landmark_dataset.pkl")

print(df_lower.head())
print(df_upper.head())



  patient_id                                           obj_path  split    jaw  \
0   00OMSZGW  __data__\3DTeethLand_combined\lower\00OMSZGW\0...  train  lower   
1   01328DDN  __data__\3DTeethLand_combined\lower\01328DDN\0...  train  lower   
2   0132CR0A  __data__\3DTeethLand_combined\lower\0132CR0A\0...   test  lower   
3   01343APK  __data__\3DTeethLand_combined\lower\01343APK\0...  train  lower   
4   01346914  __data__\3DTeethLand_combined\lower\01346914\0...  train  lower   

                                              labels  \
0  [36, 0, 0, 47, 0, 0, 35, 0, 0, 0, 0, 0, 0, 0, ...   
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 37,...   
2  [35, 43, 0, 44, 0, 0, 0, 31, 0, 45, 45, 32, 41...   
3  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 36, 45, 44, 0, ...   
4  [0, 0, 36, 32, 0, 0, 0, 0, 46, 36, 0, 31, 36, ...   

                                           instances  
0  [9, 0, 0, 4, 0, 0, 12, 0, 0, 0, 0, 0, 0, 0, 0,...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, ...  
2  [5, 10, 

## Segmentation des dents + calcul des axes / barycentres

In [ ]:
class JawPointCloudDataset(Dataset):
    def __init__(self, df, split="train", n_points=4096, preload=True):
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.n_points = n_points
        self.preload = preload

        if preload:
            self.vertices_list = []
            self.labels_list = []
            for _, row in self.df.iterrows():
                mesh = trimesh.load_mesh(row["obj_path"], process=False)
                vertices = np.asarray(mesh.vertices, dtype=np.float32)   # (N, 3)
                labels = np.asarray(row["labels"], dtype=np.int64)       # (N,)

                assert vertices.shape[0] == labels.shape[0]

                # normalisation une seule fois
                center = vertices.mean(axis=0)
                vertices = vertices - center
                scale = np.max(np.linalg.norm(vertices, axis=1))
                vertices = vertices / (scale + 1e-8)

                self.vertices_list.append(vertices)
                self.labels_list.append(labels)
        else:
            self.vertices_list = None
            self.labels_list = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.preload:
            vertices = self.vertices_list[idx]
            labels = self.labels_list[idx]
        else:
            row = self.df.iloc[idx]
            mesh = trimesh.load_mesh(row["obj_path"], process=False)
            vertices = np.asarray(mesh.vertices, dtype=np.float32)
            labels = np.asarray(row["labels"], dtype=np.int64)

            assert vertices.shape[0] == labels.shape[0]

            center = vertices.mean(axis=0)
            vertices = vertices - center
            scale = np.max(np.linalg.norm(vertices, axis=1))
            vertices = vertices / (scale + 1e-8)

        N = vertices.shape[0]
        if N >= self.n_points:
            idxs = np.random.choice(N, self.n_points, replace=False)
        else:
            idxs = np.random.choice(N, self.n_points, replace=True)

        pts = vertices[idxs]      # (n_points, 3)
        lbls = labels[idxs]       # (n_points,)

        pts = torch.from_numpy(pts)
        lbls = torch.from_numpy(lbls)

        return pts, lbls

In [ ]:
train_dataset = JawPointCloudDataset(pd.concat([df_lower, df_upper]), split="train", n_points=4096, preload=True)
test_dataset  = JawPointCloudDataset(pd.concat([df_lower, df_upper]), split="test",  n_points=4096, preload=True)
print(len(train_dataset),len(test_dataset))

## Détection des points caractéristiques (cusps, vallées, bords)

In [ ]:
dsets = torch.load(Path("__data__/jaw_pointcloud_datasets.pt"),weights_only=False)
train_dataset = dsets["train"]
test_dataset = dsets["test"]

In [8]:
# Construire le vocabulaire des labels (FDI + gencive=0)
all_labels = np.concatenate([np.concatenate(df_lower["labels"].values),
                             np.concatenate(df_upper["labels"].values)])
unique_labels = np.unique(all_labels)
fdi_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
idx_to_fdi = {i: lbl for lbl, i in fdi_to_idx.items()}

num_classes = len(unique_labels)
print("Nombre de classes :", num_classes)
print("Mapping FDI -> index :", fdi_to_idx)


Nombre de classes : 33
Mapping FDI -> index : {np.int64(0): 0, np.int64(11): 1, np.int64(12): 2, np.int64(13): 3, np.int64(14): 4, np.int64(15): 5, np.int64(16): 6, np.int64(17): 7, np.int64(18): 8, np.int64(21): 9, np.int64(22): 10, np.int64(23): 11, np.int64(24): 12, np.int64(25): 13, np.int64(26): 14, np.int64(27): 15, np.int64(28): 16, np.int64(31): 17, np.int64(32): 18, np.int64(33): 19, np.int64(34): 20, np.int64(35): 21, np.int64(36): 22, np.int64(37): 23, np.int64(38): 24, np.int64(41): 25, np.int64(42): 26, np.int64(43): 27, np.int64(44): 28, np.int64(45): 29, np.int64(46): 30, np.int64(47): 31, np.int64(48): 32}


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Trajectoire principale de l’arcade + trajectoire IOS

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TNet(nn.Module):
    def __init__(self, k: int = 3):
        super().__init__()
        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn3   = nn.BatchNorm1d(1024)

        self.fc1 = nn.Linear(1024, 512)
        self.bn4 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn5 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, k * k)

        nn.init.zeros_(self.fc3.weight)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        # x : (B, k, N)
        b, k, n = x.shape
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x, _ = torch.max(x, 2)              # (B, 1024)
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)                     # (B, k*k)

        identity = torch.eye(self.k, device=x.device).view(1, self.k * self.k)
        x = x + identity
        x = x.view(-1, self.k, self.k)      # (B, k, k)
        return x


class PointNetSegV2(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.num_classes = num_classes

        # T-Net d'entrée (3D)
        self.input_tnet = TNet(k=3)

        # MLP local 1 plus large
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2   = nn.BatchNorm1d(128)

        # T-Net de features (128D)
        self.feature_tnet = TNet(k=128)

        # MLP global plus profond
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.bn3   = nn.BatchNorm1d(256)
        self.conv4 = nn.Conv1d(256, 512, 1)
        self.bn4   = nn.BatchNorm1d(512)
        self.conv5 = nn.Conv1d(512, 1024, 1)
        self.bn5   = nn.BatchNorm1d(1024)

        # Head de segmentation (local 128 + global 1024 = 1152)
        self.conv_f1 = nn.Conv1d(1152, 512, 1)
        self.bn_f1   = nn.BatchNorm1d(512)
        self.conv_f2 = nn.Conv1d(512, 256, 1)
        self.bn_f2   = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.5)
        self.conv_f3 = nn.Conv1d(256, num_classes, 1)

    def forward(self, x):
        """
        x : (B, N, 3)
        """
        b, n, _ = x.shape
        x = x.transpose(1, 2)              # (B, 3, N)

        # T-Net d'entrée
        trans = self.input_tnet(x)         # (B, 3, 3)
        x = torch.bmm(trans, x)           # (B, 3, N)

        # MLP local 1
        x = F.relu(self.bn1(self.conv1(x)))   # (B, 64, N)
        x = F.relu(self.bn2(self.conv2(x)))   # (B, 128, N)
        pointfeat = x                          # (B, 128, N)

        # T-Net de features
        trans_feat = self.feature_tnet(x)      # (B, 128, 128)
        x = torch.bmm(trans_feat, x)           # (B, 128, N)

        # MLP global
        x = F.relu(self.bn3(self.conv3(x)))    # (B, 256, N)
        x = F.relu(self.bn4(self.conv4(x)))    # (B, 512, N)
        x = F.relu(self.bn5(self.conv5(x)))    # (B, 1024, N)

        # max-pooling global
        x_global, _ = torch.max(x, 2, keepdim=True)  # (B, 1024, 1)
        x_global = x_global.repeat(1, 1, n)          # (B, 1024, N)

        # concat local + global
        x = torch.cat([pointfeat, x_global], dim=1)  # (B, 1152, N)

        # head segmentation
        x = F.relu(self.bn_f1(self.conv_f1(x)))      # (B, 512, N)
        x = F.relu(self.bn_f2(self.conv_f2(x)))      # (B, 256, N)
        x = self.dropout(x)
        x = self.conv_f3(x)                          # (B, C, N)

        x = x.transpose(1, 2).contiguous()           # (B, N, C)
        return x


In [ ]:
num_classes = len(fdi_to_idx)
model = PointNetSegV2(num_classes=num_classes).to(device)



In [ ]:
# === Cellule 4 : DataLoaders, loss, training loop ===

batch_size = 8
num_epochs = 70
lr = 1e-3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


IGNORE_INDEX = -100  # valeur spéciale pour ignorer la gencive

# unique_labels = ensemble des FDI présents dans le dataset (comme tu faisais déjà)
max_fdi = int(max(unique_labels))

# Par défaut tout est ignoré
label_map = torch.full(
    (max_fdi + 1,),
    IGNORE_INDEX,
    dtype=torch.long,
    device=device,
)

# On mappe tous les FDI présents dans fdi_to_idx SAUF la gencive (FDI = 0)
for fdi, idx in fdi_to_idx.items():
    if int(fdi) == 0:
        continue  # gencive -> IGNORE_INDEX
    label_map[int(fdi)] = idx  # 0..num_classes-1

def encode_labels_torch(lbls_tensor):
    """
    lbls_tensor : (B, N) avec FDI (0, 11, 12, ...)
    renvoie : (B, N) avec indices de classes ou IGNORE_INDEX pour la gencive
    """
    lbls_tensor = lbls_tensor.to(device).long()
    return label_map[lbls_tensor]


num_classes = len(fdi_to_idx)  # tu peux garder tel quel (classe "gencive" inutilisée)

# poids = 1 pour toutes les classes par défaut
class_weights = torch.ones(num_classes, device=device)

# booster les troisièmes molaires
third_molars = [18, 28, 38, 48]
for fdi in third_molars:
    idx = fdi_to_idx.get(fdi, None)
    if idx is not None:
        class_weights[idx] = 3.0  # ou 2.0, à tester

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX
)


optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# scheduler : si test_acc ne monte plus, on divise lr par 2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",      # on veut maximiser test_acc
    factor=0.5,      # lr = lr * 0.5
    patience=3,      # après 5 epochs sans amélioration
    min_lr=1e-5
)

best_test_acc = 0.0
best_state = None

mirror_map = torch.arange(max_fdi + 1, device=device)  # identité par défaut

pairs = [
    (11, 21), (12, 22), (13, 23), (14, 24), (15, 25), (16, 26), (17, 27), (18, 28),
    (31, 41), (32, 42), (33, 43), (34, 44), (35, 45), (36, 46), (37, 47), (38, 48),
]

for a, b in pairs:
    mirror_map[a] = b
    mirror_map[b] = a
# 0 (gencive) reste mappé sur lui‑même


flip_prob = 0.5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_points_teeth = 0
    correct_teeth = 0

    for pts, lbls_fdi in train_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) FDI
        pts = pts.to(device)
        lbls_fdi = lbls_fdi.to(device).long()

        # ---- augmentation symétrie gauche/droite ----
        if torch.rand(1).item() < flip_prob:
            # flip X
            pts[..., 0] = -pts[..., 0]
            # remap FDI
            lbls_fdi = mirror_map[lbls_fdi]

        # encode FDI -> indices (avec IGNORE_INDEX pour la gencive)
        lbls_idx = encode_labels_torch(lbls_fdi)  # (B, N)

        optimizer.zero_grad()
        logits = model(pts)   # (B, N, C)

        loss = criterion(
            logits.view(-1, num_classes),
            lbls_idx.view(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * pts.shape[0]

        preds = logits.argmax(dim=-1)
        mask = (lbls_idx != IGNORE_INDEX)
        correct_teeth += (preds[mask] == lbls_idx[mask]).sum().item()
        total_points_teeth += mask.sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc_teeth = correct_teeth / total_points_teeth


    # --- évaluation sur test ---
    model.eval()
    total_points_teeth_test = 0
    correct_teeth_test = 0

    with torch.no_grad():
        for pts, lbls_fdi in test_loader:
            pts = pts.to(device)
            lbls_idx = encode_labels_torch(lbls_fdi).to(device)

            logits = model(pts)
            preds = logits.argmax(dim=-1)

            mask = (lbls_idx != IGNORE_INDEX)
            correct_teeth_test += (preds[mask] == lbls_idx[mask]).sum().item()
            total_points_teeth_test += mask.sum().item()

    test_acc_teeth = correct_teeth_test / total_points_teeth_test

    print(
        f"[Epoch {epoch+1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} | "
        f"Train acc (teeth only): {train_acc_teeth:.3f} | "
        f"Test acc (teeth only): {test_acc_teeth:.3f}"
    )

    scheduler.step(test_acc_teeth)


[Epoch 1/70] Loss: 1.6459 | Train acc (teeth only): 0.466 | Test acc (teeth only): 0.622
[Epoch 2/70] Loss: 0.9856 | Train acc (teeth only): 0.643 | Test acc (teeth only): 0.685
[Epoch 3/70] Loss: 0.8949 | Train acc (teeth only): 0.670 | Test acc (teeth only): 0.548
[Epoch 4/70] Loss: 0.8276 | Train acc (teeth only): 0.690 | Test acc (teeth only): 0.636
[Epoch 5/70] Loss: 0.8210 | Train acc (teeth only): 0.696 | Test acc (teeth only): 0.685
[Epoch 6/70] Loss: 0.7479 | Train acc (teeth only): 0.712 | Test acc (teeth only): 0.714
[Epoch 7/70] Loss: 0.7992 | Train acc (teeth only): 0.708 | Test acc (teeth only): 0.389
[Epoch 8/70] Loss: 0.7163 | Train acc (teeth only): 0.724 | Test acc (teeth only): 0.681
[Epoch 9/70] Loss: 0.7863 | Train acc (teeth only): 0.712 | Test acc (teeth only): 0.674
[Epoch 10/70] Loss: 0.7539 | Train acc (teeth only): 0.721 | Test acc (teeth only): 0.660
[Epoch 11/70] Loss: 0.6592 | Train acc (teeth only): 0.753 | Test acc (teeth only): 0.791
[Epoch 12/70] Loss:

In [ ]:
# Construire idx -> FDI si tu ne l'as pas déjà
idx_to_fdi = {idx: fdi for fdi, idx in fdi_to_idx.items()}
num_classes = len(idx_to_fdi)

correct_per_class = torch.zeros(num_classes, dtype=torch.long)
total_per_class   = torch.zeros(num_classes, dtype=torch.long)

model.eval()
with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) avec FDI
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)              # (B, N, C)
        preds_idx = logits.argmax(dim=-1)  # (B, N)

        for c in range(num_classes):
            mask = (lbls_idx == c)
            if mask.any():
                total_per_class[c]   += mask.sum().item()
                correct_per_class[c] += (preds_idx[mask] == c).sum().item()

# Affichage des accuracies par FDI
for c in range(num_classes):
    if total_per_class[c] == 0:
        continue
    fdi = idx_to_fdi[c]
    acc = correct_per_class[c].item() / total_per_class[c].item()
    print(f"FDI {fdi}: acc = {acc:.3f} (N = {total_per_class[c].item()})")


FDI 11: acc = 0.925 (N = 47179)
FDI 12: acc = 0.882 (N = 33406)
FDI 13: acc = 0.891 (N = 31943)
FDI 14: acc = 0.931 (N = 41914)
FDI 15: acc = 0.855 (N = 43750)
FDI 16: acc = 0.888 (N = 72391)
FDI 17: acc = 0.814 (N = 36677)
FDI 18: acc = 0.874 (N = 1766)
FDI 21: acc = 0.924 (N = 48724)
FDI 22: acc = 0.878 (N = 34140)
FDI 23: acc = 0.879 (N = 32945)
FDI 24: acc = 0.934 (N = 42832)
FDI 25: acc = 0.864 (N = 43563)
FDI 26: acc = 0.881 (N = 72048)
FDI 27: acc = 0.817 (N = 38480)
FDI 28: acc = 0.901 (N = 1500)
FDI 31: acc = 0.825 (N = 36920)
FDI 32: acc = 0.825 (N = 39717)
FDI 33: acc = 0.833 (N = 44658)
FDI 34: acc = 0.822 (N = 49209)
FDI 35: acc = 0.751 (N = 57310)
FDI 36: acc = 0.858 (N = 93987)
FDI 37: acc = 0.828 (N = 52462)
FDI 38: acc = 0.727 (N = 2559)
FDI 41: acc = 0.825 (N = 36729)
FDI 42: acc = 0.821 (N = 39802)
FDI 43: acc = 0.846 (N = 45007)
FDI 44: acc = 0.840 (N = 51366)
FDI 45: acc = 0.772 (N = 56529)
FDI 46: acc = 0.861 (N = 92128)
FDI 47: acc = 0.768 (N = 50550)
FDI 48: acc

In [ ]:
model.eval()
total_tooth = 0
correct_tooth = 0

with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)
        preds_idx = logits.argmax(dim=-1)                    # (B, N)

        # même masque que dans la boucle d'entraînement
        mask = (lbls_idx != IGNORE_INDEX)

        correct_tooth += (preds_idx[mask] == lbls_idx[mask]).sum().item()
        total_tooth += mask.sum().item()

tooth_acc = correct_tooth / total_tooth
print("Tooth-only accuracy (FDI != 0):", tooth_acc)



Tooth-only accuracy (FDI != 0): 0.8506732854335964


In [ ]:
from pathlib import Path

SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

ckpt_path = SAVE_DIR / "PointNetSeg_no_gencive.pt"

torch.save({
    "model_state": model.state_dict(),
    "fdi_to_idx": fdi_to_idx,
    "idx_to_fdi": idx_to_fdi,
}, ckpt_path)
print("Checkpoint sauvé dans :", ckpt_path)


Checkpoint sauvé dans : checkpoints\PointNetSeg_no_gencive.pt


In [ ]:
ckpt_path = Path("checkpoints") / "PointNetSeg_no_gencive.pt"
ckpt = torch.load(ckpt_path, map_location=device,weights_only=False)

model = PointNetSegV2(num_classes=len(ckpt["fdi_to_idx"])).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

fdi_to_idx = ckpt["fdi_to_idx"]
idx_to_fdi = ckpt["idx_to_fdi"]

In [ ]:
max_fdi = int(max(unique_labels))
batch_size=8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


IGNORE_INDEX = -100  # valeur spéciale pour ignorer la gencive

def encode_labels_torch(lbls_tensor):
    """
    lbls_tensor : (B, N) avec FDI (0, 11, 12, ...)
    renvoie : (B, N) avec indices de classes ou IGNORE_INDEX pour la gencive
    """
    lbls_tensor = lbls_tensor.to(device).long()
    return label_map[lbls_tensor]

# Par défaut tout est ignoré
label_map = torch.full(
    (max_fdi + 1,),
    IGNORE_INDEX,
    dtype=torch.long,
    device=device,
)

# On mappe tous les FDI présents dans fdi_to_idx SAUF la gencive (FDI = 0)
for fdi, idx in fdi_to_idx.items():
    if int(fdi) == 0:
        continue  # gencive -> IGNORE_INDEX
    label_map[int(fdi)] = idx  # 0..num_classes-1

def encode_labels_torch(lbls_tensor):
    """
    lbls_tensor : (B, N) avec FDI (0, 11, 12, ...)
    renvoie : (B, N) avec indices de classes ou IGNORE_INDEX pour la gencive
    """
    lbls_tensor = lbls_tensor.to(device).long()
    return label_map[lbls_tensor]

In [ ]:
import collections
import torch

model.eval()

target_fdi = 48
target_idx = fdi_to_idx[target_fdi]

pred_counts_48 = collections.Counter()

with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) en FDI
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)
        logits = model(pts)                                  # (B, N, C)
        preds_idx = logits.argmax(dim=-1)                    # (B, N)

        # masque pour les points dont le vrai label est 48
        mask_48 = (lbls_idx == target_idx)
        if mask_48.any():
            preds_48 = preds_idx[mask_48].cpu().numpy()
            for p in preds_48:
                pred_counts_48[int(p)] += 1

print(f"FDI {target_fdi} (class idx {target_idx}) predictions en test :")
for idx, cnt in pred_counts_48.items():
    print(f"  prédit comme class idx {idx} (FDI {idx_to_fdi[idx]}) : {cnt} points")


FDI 48 (class idx 32) predictions en test :
  prédit comme class idx 32 (FDI 48) : 1689 points
  prédit comme class idx 31 (FDI 47) : 961 points
  prédit comme class idx 30 (FDI 46) : 48 points


In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset
import trimesh

def compute_arc_index_per_vertex(vertices: np.ndarray,
                                 instances: np.ndarray,
                                 jaw: str,
                                 axis: int = 0) -> np.ndarray:
    """
    vertices : (V,3), instances : (V,), jaw : 'upper' / 'lower'
    Retourne t_full : (V,) dans [0,1], indice d'arcade par vertex.
    """
    verts = vertices
    inst = instances

    inst_ids = sorted(int(i) for i in np.unique(inst) if i != 0)
    if len(inst_ids) == 0:
        return np.full((verts.shape[0],), 0.5, dtype=np.float32)

    centers = []
    for iid in inst_ids:
        mask = (inst == iid)
        if not np.any(mask):
            continue
        centers.append(verts[mask].mean(axis=0))
    centers = np.array(centers)
    inst_ids = np.array(inst_ids)

    order = np.argsort(centers[:, axis])  # gauche->droite (ou inverse selon repère)
    inst_ids_ordered = inst_ids[order]

    n = len(inst_ids_ordered)
    if n > 1:
        t_values = np.linspace(0.0, 1.0, n, dtype=np.float32)
    else:
        t_values = np.array([0.5], dtype=np.float32)

    t_per_inst = {int(iid): float(t) for iid, t in zip(inst_ids_ordered, t_values)}

    t_full = np.zeros((verts.shape[0],), dtype=np.float32)
    for iid, t in t_per_inst.items():
        mask = (inst == iid)
        t_full[mask] = t

    # gencive -> valeur neutre
    t_full[inst == 0] = 0.5
    return t_full


class JawPointCloudDatasetArc(Dataset):
    def __init__(self, df, split="train", n_points=4096, preload=True):
        """
        df      : DataFrame concat(df_lower, df_upper)
        split   : 'train' / 'test'
        n_points: nb de points échantillonnés par cas
        preload : si True, charge et pré-calcul tout en mémoire
        """
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.n_points = n_points
        self.preload = preload

        if preload:
            self.vertices_list = []
            self.labels_list = []
            self.t_list = []

            for _, row in self.df.iterrows():
                jaw = row["jaw"]  # 'upper' ou 'lower'
                mesh = trimesh.load_mesh(row["obj_path"], process=False)
                vertices = np.asarray(mesh.vertices, dtype=np.float32)       # (V,3)
                labels   = np.asarray(row["labels"], dtype=np.int64)         # (V,)
                instances = np.asarray(row["instances"], dtype=np.int64)     # (V,)

                # t avant ou après normalisation, peu importe (relatif)
                t_full = compute_arc_index_per_vertex(vertices, instances, jaw, axis=0)

                # normalisation comme dans ton dataset original
                center = vertices.mean(axis=0)
                vertices = vertices - center
                scale = np.max(np.linalg.norm(vertices, axis=1))
                vertices = vertices / (scale + 1e-8)

                self.vertices_list.append(vertices)   # (V,3)
                self.labels_list.append(labels)       # (V,)
                self.t_list.append(t_full)            # (V,)
        else:
            self.vertices_list = None
            self.labels_list = None
            self.t_list = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.preload:
            vertices = self.vertices_list[idx]
            labels   = self.labels_list[idx]
            t_full   = self.t_list[idx]
        else:
            row = self.df.iloc[idx]
            jaw = row["jaw"]
            mesh = trimesh.load_mesh(row["obj_path"], process=False)
            vertices = np.asarray(mesh.vertices, dtype=np.float32)
            labels   = np.asarray(row["labels"], dtype=np.int64)
            instances = np.asarray(row["instances"], dtype=np.int64)

            t_full = compute_arc_index_per_vertex(vertices, instances, jaw, axis=0)

            center = vertices.mean(axis=0)
            vertices = vertices - center
            scale = np.max(np.linalg.norm(vertices, axis=1))
            vertices = vertices / (scale + 1e-8)

        V = vertices.shape[0]
        if V >= self.n_points:
            idxs = np.random.choice(V, self.n_points, replace=False)
        else:
            idxs = np.random.choice(V, self.n_points, replace=True)

        pts = vertices[idxs]              # (N,3)
        t   = t_full[idxs][:, None]       # (N,1)
        pts4 = np.concatenate([pts, t], axis=1)  # (N,4)
        lbls = labels[idxs]               # (N,)

        pts4 = torch.from_numpy(pts4).float()
        lbls = torch.from_numpy(lbls).long()
        return pts4, lbls


In [ ]:
df_all = pd.concat([df_lower, df_upper], ignore_index=True)

train_dataset = JawPointCloudDatasetArc(df_all, split="train", n_points=4096)
test_dataset  = JawPointCloudDatasetArc(df_all, split="test",  n_points=4096)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TNet(nn.Module):
    def __init__(self, k: int = 3):
        super().__init__()
        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn3   = nn.BatchNorm1d(1024)

        self.fc1 = nn.Linear(1024, 512)
        self.bn4 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn5 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, k * k)

        nn.init.zeros_(self.fc3.weight)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        # x : (B, k, N)
        b, k, n = x.shape

        x = F.relu(self.bn1(self.conv1(x)))   # (B, 64, N)
        x = F.relu(self.bn2(self.conv2(x)))   # (B, 128, N)
        x = F.relu(self.bn3(self.conv3(x)))   # (B, 1024, N)

        x = torch.max(x, 2)[0]               # (B, 1024)
        x = F.relu(self.bn4(self.fc1(x)))    # (B, 512)
        x = F.relu(self.bn5(self.fc2(x)))    # (B, 256)
        x = self.fc3(x)                      # (B, k*k)

        ident = torch.eye(self.k, device=x.device).view(1, self.k * self.k)
        x = x + ident
        x = x.view(-1, self.k, self.k)       # (B, k, k)
        return x


class PointNetSegXYZT(nn.Module):
    """
    PointNet segmentation:
      - T-Net + MLP sur xyz (3D)
      - 't' injecté dans le head de segmentation
    Entrée:  x : (B, N, 4) -> (x,y,z,t)
    Sortie: logits : (B, N, num_classes)
    """
    def __init__(self, num_classes: int, dropout: float = 0.4):
        super().__init__()
        self.num_classes = num_classes

        # T-Net 3D
        self.input_tnet = TNet(k=3)

        # MLP local sur xyz
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn3   = nn.BatchNorm1d(1024)

        # Head de segmentation (local 128 + global 1024 + t (1) = 1153)
        self.conv_seg1 = nn.Conv1d(1153, 512, 1)
        self.bn_seg1   = nn.BatchNorm1d(512)
        self.conv_seg2 = nn.Conv1d(512, 256, 1)
        self.bn_seg2   = nn.BatchNorm1d(256)
        self.dropout   = nn.Dropout(dropout)
        self.conv_seg3 = nn.Conv1d(256, num_classes, 1)

    def forward(self, x):
        """
        x : (B, N, 4)  avec x[..., :3] = xyz, x[..., 3] = t
        """
        B, N, _ = x.shape

        xyz = x[..., :3].transpose(1, 2).contiguous()   # (B, 3, N)
        t   = x[..., 3:].transpose(1, 2).contiguous()   # (B, 1, N)

        # T-Net sur xyz uniquement
        trans = self.input_tnet(xyz)                    # (B, 3, 3)
        xyz = torch.bmm(trans, xyz)                     # (B, 3, N)

        # MLP local
        x1 = F.relu(self.bn1(self.conv1(xyz)))          # (B, 64, N)
        x2 = F.relu(self.bn2(self.conv2(x1)))           # (B, 128, N)
        pointfeat = x2                                  # (B, 128, N)

        # Global feature
        x3 = F.relu(self.bn3(self.conv3(x2)))           # (B, 1024, N)
        global_feat = torch.max(x3, 2, keepdim=True)[0] # (B, 1024, 1)
        global_feat = global_feat.repeat(1, 1, N)       # (B, 1024, N)

        # Fusion local + global + t
        feat = torch.cat([pointfeat, global_feat, t], dim=1)  # (B, 128+1024+1=1153, N)

        # Head de segmentation
        feat = F.relu(self.bn_seg1(self.conv_seg1(feat)))     # (B, 512, N)
        feat = F.relu(self.bn_seg2(self.conv_seg2(feat)))     # (B, 256, N)
        feat = self.dropout(feat)
        feat = self.conv_seg3(feat)                           # (B, C, N)

        return feat.transpose(1, 2).contiguous()              # (B, N, C)


In [ ]:
num_classes = len(fdi_to_idx)
model = PointNetSegXYZT(num_classes=num_classes).to(device)


In [ ]:
# === Cellule 4 : DataLoaders, loss, training loop ===

batch_size = 8
num_epochs = 70
lr = 1e-3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


IGNORE_INDEX = -100  # valeur spéciale pour ignorer la gencive

# unique_labels = ensemble des FDI présents dans le dataset (comme tu faisais déjà)
max_fdi = int(max(unique_labels))

# Par défaut tout est ignoré
label_map = torch.full(
    (max_fdi + 1,),
    IGNORE_INDEX,
    dtype=torch.long,
    device=device,
)

# On mappe tous les FDI présents dans fdi_to_idx SAUF la gencive (FDI = 0)
for fdi, idx in fdi_to_idx.items():
    if int(fdi) == 0:
        continue  # gencive -> IGNORE_INDEX
    label_map[int(fdi)] = idx  # 0..num_classes-1

def encode_labels_torch(lbls_tensor):
    """
    lbls_tensor : (B, N) avec FDI (0, 11, 12, ...)
    renvoie : (B, N) avec indices de classes ou IGNORE_INDEX pour la gencive
    """
    lbls_tensor = lbls_tensor.to(device).long()
    return label_map[lbls_tensor]


num_classes = len(fdi_to_idx)  # tu peux garder tel quel (classe "gencive" inutilisée)

# poids = 1 pour toutes les classes par défaut
class_weights = torch.ones(num_classes, device=device)

# booster les troisièmes molaires
third_molars = [18, 28, 38, 48]
for fdi in third_molars:
    idx = fdi_to_idx.get(fdi, None)
    if idx is not None:
        class_weights[idx] = 3.0  # ou 2.0, à tester

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX
)


optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# scheduler : si test_acc ne monte plus, on divise lr par 2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",      # on veut maximiser test_acc
    factor=0.5,      # lr = lr * 0.5
    patience=3,      # après 5 epochs sans amélioration
    min_lr=1e-5
)

best_test_acc = 0.0
best_state = None

mirror_map = torch.arange(max_fdi + 1, device=device)  # identité par défaut

pairs = [
    (11, 21), (12, 22), (13, 23), (14, 24), (15, 25), (16, 26), (17, 27), (18, 28),
    (31, 41), (32, 42), (33, 43), (34, 44), (35, 45), (36, 46), (37, 47), (38, 48),
]

for a, b in pairs:
    mirror_map[a] = b
    mirror_map[b] = a
# 0 (gencive) reste mappé sur lui‑même


flip_prob = 0.5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_points_teeth = 0
    correct_teeth = 0

    for pts, lbls_fdi in train_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) FDI
        pts = pts.to(device)
        lbls_fdi = lbls_fdi.to(device).long()

        # ---- augmentation symétrie gauche/droite ----
        if torch.rand(1).item() < flip_prob:
            # flip X
            pts[..., 0] = -pts[..., 0]
            # remap FDI
            lbls_fdi = mirror_map[lbls_fdi]

        # encode FDI -> indices (avec IGNORE_INDEX pour la gencive)
        lbls_idx = encode_labels_torch(lbls_fdi)  # (B, N)

        optimizer.zero_grad()
        logits = model(pts)   # (B, N, C)

        loss = criterion(
            logits.view(-1, num_classes),
            lbls_idx.view(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * pts.shape[0]

        preds = logits.argmax(dim=-1)
        mask = (lbls_idx != IGNORE_INDEX)
        correct_teeth += (preds[mask] == lbls_idx[mask]).sum().item()
        total_points_teeth += mask.sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc_teeth = correct_teeth / total_points_teeth


    # --- évaluation sur test ---
    model.eval()
    total_points_teeth_test = 0
    correct_teeth_test = 0

    with torch.no_grad():
        for pts, lbls_fdi in test_loader:
            pts = pts.to(device)
            lbls_idx = encode_labels_torch(lbls_fdi).to(device)

            logits = model(pts)
            preds = logits.argmax(dim=-1)

            mask = (lbls_idx != IGNORE_INDEX)
            correct_teeth_test += (preds[mask] == lbls_idx[mask]).sum().item()
            total_points_teeth_test += mask.sum().item()

    test_acc_teeth = correct_teeth_test / total_points_teeth_test

    print(
        f"[Epoch {epoch+1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} | "
        f"Train acc (teeth only): {train_acc_teeth:.3f} | "
        f"Test acc (teeth only): {test_acc_teeth:.3f}"
    )

    scheduler.step(test_acc_teeth)


[Epoch 1/70] Loss: 1.2729 | Train acc (teeth only): 0.578 | Test acc (teeth only): 0.586
[Epoch 2/70] Loss: 0.8133 | Train acc (teeth only): 0.695 | Test acc (teeth only): 0.475
[Epoch 3/70] Loss: 0.7947 | Train acc (teeth only): 0.702 | Test acc (teeth only): 0.613
[Epoch 4/70] Loss: 0.7224 | Train acc (teeth only): 0.725 | Test acc (teeth only): 0.634
[Epoch 5/70] Loss: 0.6968 | Train acc (teeth only): 0.732 | Test acc (teeth only): 0.301
[Epoch 6/70] Loss: 0.7343 | Train acc (teeth only): 0.719 | Test acc (teeth only): 0.520
[Epoch 7/70] Loss: 0.6600 | Train acc (teeth only): 0.744 | Test acc (teeth only): 0.583
[Epoch 8/70] Loss: 0.6500 | Train acc (teeth only): 0.749 | Test acc (teeth only): 0.702
[Epoch 9/70] Loss: 0.6307 | Train acc (teeth only): 0.758 | Test acc (teeth only): 0.352
[Epoch 10/70] Loss: 0.6378 | Train acc (teeth only): 0.756 | Test acc (teeth only): 0.655
[Epoch 11/70] Loss: 0.5950 | Train acc (teeth only): 0.772 | Test acc (teeth only): 0.745
[Epoch 12/70] Loss:

In [ ]:
# Construire idx -> FDI si tu ne l'as pas déjà
idx_to_fdi = {idx: fdi for fdi, idx in fdi_to_idx.items()}
num_classes = len(idx_to_fdi)

correct_per_class = torch.zeros(num_classes, dtype=torch.long)
total_per_class   = torch.zeros(num_classes, dtype=torch.long)

model.eval()
with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) avec FDI
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)              # (B, N, C)
        preds_idx = logits.argmax(dim=-1)  # (B, N)

        for c in range(num_classes):
            mask = (lbls_idx == c)
            if mask.any():
                total_per_class[c]   += mask.sum().item()
                correct_per_class[c] += (preds_idx[mask] == c).sum().item()

# Affichage des accuracies par FDI
for c in range(num_classes):
    if total_per_class[c] == 0:
        continue
    fdi = idx_to_fdi[c]
    acc = correct_per_class[c].item() / total_per_class[c].item()
    print(f"FDI {fdi}: acc = {acc:.3f} (N = {total_per_class[c].item()})")

FDI 11: acc = 0.926 (N = 47206)
FDI 12: acc = 0.867 (N = 33553)
FDI 13: acc = 0.887 (N = 32083)
FDI 14: acc = 0.900 (N = 41873)
FDI 15: acc = 0.805 (N = 43934)
FDI 16: acc = 0.870 (N = 72553)
FDI 17: acc = 0.821 (N = 36643)
FDI 18: acc = 0.828 (N = 1735)
FDI 21: acc = 0.922 (N = 48380)
FDI 22: acc = 0.887 (N = 34422)
FDI 23: acc = 0.887 (N = 32990)
FDI 24: acc = 0.912 (N = 42655)
FDI 25: acc = 0.834 (N = 43981)
FDI 26: acc = 0.882 (N = 72505)
FDI 27: acc = 0.872 (N = 38502)
FDI 28: acc = 0.862 (N = 1509)
FDI 31: acc = 0.855 (N = 36823)
FDI 32: acc = 0.847 (N = 40130)
FDI 33: acc = 0.840 (N = 44391)
FDI 34: acc = 0.847 (N = 49184)
FDI 35: acc = 0.788 (N = 57350)
FDI 36: acc = 0.848 (N = 94780)
FDI 37: acc = 0.827 (N = 52745)
FDI 38: acc = 0.774 (N = 2598)
FDI 41: acc = 0.852 (N = 36292)
FDI 42: acc = 0.846 (N = 39747)
FDI 43: acc = 0.861 (N = 45090)
FDI 44: acc = 0.850 (N = 51151)
FDI 45: acc = 0.795 (N = 56396)
FDI 46: acc = 0.867 (N = 91905)
FDI 47: acc = 0.870 (N = 49838)
FDI 48: acc

In [ ]:
model.eval()
total_tooth = 0
correct_tooth = 0

with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)
        preds_idx = logits.argmax(dim=-1)                    # (B, N)

        # même masque que dans la boucle d'entraînement
        mask = (lbls_idx != IGNORE_INDEX)

        correct_tooth += (preds_idx[mask] == lbls_idx[mask]).sum().item()
        total_tooth += mask.sum().item()

tooth_acc = correct_tooth / total_tooth
print("Tooth-only accuracy (FDI != 0):", tooth_acc)



Tooth-only accuracy (FDI != 0): 0.8581291055996996


In [ ]:
from pathlib import Path

SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

ckpt_path = SAVE_DIR / "Arc_PointNetSeg_no_gencive.pt"

torch.save({
    "model_state": model.state_dict(),
    "fdi_to_idx": fdi_to_idx,
    "idx_to_fdi": idx_to_fdi,
}, ckpt_path)
print("Checkpoint sauvé dans :", ckpt_path)


Checkpoint sauvé dans : checkpoints\Arc_PointNetSeg_no_gencive.pt


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def knn(x, k):
    # x : (B, C, N)
    inner = -2 * torch.matmul(x.transpose(2, 1), x)   # (B, N, N)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)       # (B, 1, N)
    dist = xx + xx.transpose(2, 1) + inner            # (B, N, N)
    idx = dist.topk(k=k, dim=-1, largest=False)[1]    # plus proches
    return idx                                        # (B, N, k)

def get_graph_feature(x, k=20, idx=None):
    # x : (B, C, N) -> (B, 2C, N, k)
    B, C, N = x.size()
    if idx is None:
        idx = knn(x, k=k)         # (B, N, k)
    device = x.device

    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = idx + idx_base
    idx = idx.view(-1)

    x = x.transpose(2, 1).contiguous()      # (B, N, C)
    feature = x.view(B * N, C)[idx, :]
    feature = feature.view(B, N, k, C)      # (B, N, k, C)

    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)   # (B, N, k, 2C)
    feature = feature.permute(0, 3, 1, 2).contiguous()  # (B, 2C, N, k)
    return feature

class DGCNNSeg4D(nn.Module):
    def __init__(self, num_classes: int, k: int = 20, emb_dims: int = 1024, dropout: float = 0.5):
        super().__init__()
        self.k = k

        # C_in = 4 -> 2C = 8
        self.conv1 = nn.Sequential(
            nn.Conv2d(8, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.conv5 = nn.Sequential(
            nn.Conv1d(512, emb_dims, kernel_size=1, bias=False),
            nn.BatchNorm1d(emb_dims),
            nn.ReLU(inplace=True)
        )

        self.conv6 = nn.Sequential(
            nn.Conv1d(512 + emb_dims, 512, kernel_size=1, bias=False),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True)
        )
        self.conv7 = nn.Sequential(
            nn.Conv1d(512, 256, kernel_size=1, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True)
        )
        self.dropout = nn.Dropout(p=dropout)
        self.conv8 = nn.Conv1d(256, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        """
        x : (B, N, 4)  # (x,y,z,t)
        """
        B, N, _ = x.size()
        x = x.permute(0, 2, 1).contiguous()   # (B, 4, N)

        x1 = get_graph_feature(x, k=self.k)   # (B, 8, N, k)
        x1 = self.conv1(x1)
        x1 = x1.max(dim=-1)[0]                # (B, 64, N)

        x2 = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(x2)
        x2 = x2.max(dim=-1)[0]                # (B, 64, N)

        x3 = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(x3)
        x3 = x3.max(dim=-1)[0]                # (B, 128, N)

        x4 = get_graph_feature(x3, k=self.k)
        x4 = self.conv4(x4)
        x4 = x4.max(dim=-1)[0]                # (B, 256, N)

        x_cat = torch.cat((x1, x2, x3, x4), dim=1)  # (B, 512, N)

        x_global = self.conv5(x_cat)          # (B, emb_dims, N)
        x_max = x_global.max(dim=-1, keepdim=True)[0]    # (B, emb_dims, 1)
        x_global_expanded = x_max.repeat(1, 1, N)        # (B, emb_dims, N)

        x_final = torch.cat((x_cat, x_global_expanded), dim=1)  # (B, 512+emb_dims, N)

        x_final = self.conv6(x_final)         # (B, 512, N)
        x_final = self.conv7(x_final)         # (B, 256, N)
        x_final = self.dropout(x_final)
        x_final = self.conv8(x_final)         # (B, C, N)

        return x_final.permute(0, 2, 1).contiguous()  # (B, N, C)


In [9]:
num_classes = len(fdi_to_idx)
model = DGCNNSeg4D(num_classes=num_classes, k=20, emb_dims=1024, dropout=0.5).to(device)


In [10]:
# === Cellule 4 : DataLoaders, loss, training loop ===

batch_size = 8
num_epochs = 20
lr = 1e-3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


IGNORE_INDEX = -100  # valeur spéciale pour ignorer la gencive

# unique_labels = ensemble des FDI présents dans le dataset (comme tu faisais déjà)
max_fdi = int(max(unique_labels))

# Par défaut tout est ignoré
label_map = torch.full(
    (max_fdi + 1,),
    IGNORE_INDEX,
    dtype=torch.long,
    device=device,
)

# On mappe tous les FDI présents dans fdi_to_idx SAUF la gencive (FDI = 0)
for fdi, idx in fdi_to_idx.items():
    if int(fdi) == 0:
        continue  # gencive -> IGNORE_INDEX
    label_map[int(fdi)] = idx  # 0..num_classes-1

def encode_labels_torch(lbls_tensor):
    """
    lbls_tensor : (B, N) avec FDI (0, 11, 12, ...)
    renvoie : (B, N) avec indices de classes ou IGNORE_INDEX pour la gencive
    """
    lbls_tensor = lbls_tensor.to(device).long()
    return label_map[lbls_tensor]


num_classes = len(fdi_to_idx)  # tu peux garder tel quel (classe "gencive" inutilisée)

# poids = 1 pour toutes les classes par défaut
class_weights = torch.ones(num_classes, device=device)

# booster les troisièmes molaires
third_molars = [18, 28, 38, 48]
for fdi in third_molars:
    idx = fdi_to_idx.get(fdi, None)
    if idx is not None:
        class_weights[idx] = 3.0  # ou 2.0, à tester

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=IGNORE_INDEX
)


optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# scheduler : si test_acc ne monte plus, on divise lr par 2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",      # on veut maximiser test_acc
    factor=0.5,      # lr = lr * 0.5
    patience=3,      # après 5 epochs sans amélioration
    min_lr=1e-5
)

best_test_acc = 0.0
best_state = None

mirror_map = torch.arange(max_fdi + 1, device=device)  # identité par défaut

pairs = [
    (11, 21), (12, 22), (13, 23), (14, 24), (15, 25), (16, 26), (17, 27), (18, 28),
    (31, 41), (32, 42), (33, 43), (34, 44), (35, 45), (36, 46), (37, 47), (38, 48),
]

for a, b in pairs:
    mirror_map[a] = b
    mirror_map[b] = a
# 0 (gencive) reste mappé sur lui‑même


flip_prob = 0.5
best_test_acc = 0.0
best_state = None

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_points_teeth = 0
    correct_teeth = 0

    for pts, lbls_fdi in train_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) FDI
        pts = pts.to(device)
        lbls_fdi = lbls_fdi.to(device).long()

        # ---- augmentation symétrie gauche/droite ----
        if torch.rand(1).item() < flip_prob:
            # flip X
            pts[..., 0] = -pts[..., 0]
            # remap FDI
            lbls_fdi = mirror_map[lbls_fdi]

        # encode FDI -> indices (avec IGNORE_INDEX pour la gencive)
        lbls_idx = encode_labels_torch(lbls_fdi)  # (B, N)

        optimizer.zero_grad()
        logits = model(pts)   # (B, N, C)

        loss = criterion(
            logits.view(-1, num_classes),
            lbls_idx.view(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * pts.shape[0]

        preds = logits.argmax(dim=-1)
        mask = (lbls_idx != IGNORE_INDEX)
        correct_teeth += (preds[mask] == lbls_idx[mask]).sum().item()
        total_points_teeth += mask.sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc_teeth = correct_teeth / total_points_teeth


    # --- évaluation sur test ---
    model.eval()
    total_points_teeth_test = 0
    correct_teeth_test = 0

    with torch.no_grad():
        for pts, lbls_fdi in test_loader:
            pts = pts.to(device)
            lbls_idx = encode_labels_torch(lbls_fdi).to(device)

            logits = model(pts)
            preds = logits.argmax(dim=-1)

            mask = (lbls_idx != IGNORE_INDEX)
            correct_teeth_test += (preds[mask] == lbls_idx[mask]).sum().item()
            total_points_teeth_test += mask.sum().item()

    test_acc_teeth = correct_teeth_test / total_points_teeth_test

    print(
        f"[Epoch {epoch+1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} | "
        f"Train acc (teeth only): {train_acc_teeth:.3f} | "
        f"Test acc (teeth only): {test_acc_teeth:.3f}"
    )

    scheduler.step(test_acc_teeth)

    if test_acc_teeth > best_test_acc:
        best_test_acc = test_acc_teeth
        best_state = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "fdi_to_idx": fdi_to_idx,
            "idx_to_fdi": idx_to_fdi,
            "test_acc_teeth": best_test_acc,
        }



[Epoch 1/20] Loss: 1.3017 | Train acc (teeth only): 0.589 | Test acc (teeth only): 0.778
[Epoch 2/20] Loss: 0.7094 | Train acc (teeth only): 0.759 | Test acc (teeth only): 0.814
[Epoch 3/20] Loss: 0.5629 | Train acc (teeth only): 0.813 | Test acc (teeth only): 0.859
[Epoch 4/20] Loss: 0.5075 | Train acc (teeth only): 0.833 | Test acc (teeth only): 0.819
[Epoch 5/20] Loss: 0.4006 | Train acc (teeth only): 0.870 | Test acc (teeth only): 0.612
[Epoch 6/20] Loss: 0.3931 | Train acc (teeth only): 0.871 | Test acc (teeth only): 0.798
[Epoch 7/20] Loss: 0.3391 | Train acc (teeth only): 0.893 | Test acc (teeth only): 0.905
[Epoch 8/20] Loss: 0.3607 | Train acc (teeth only): 0.883 | Test acc (teeth only): 0.874
[Epoch 9/20] Loss: 0.2996 | Train acc (teeth only): 0.902 | Test acc (teeth only): 0.814
[Epoch 10/20] Loss: 0.2927 | Train acc (teeth only): 0.906 | Test acc (teeth only): 0.837
[Epoch 11/20] Loss: 0.2999 | Train acc (teeth only): 0.899 | Test acc (teeth only): 0.814
[Epoch 12/20] Loss:

In [11]:
# Construire idx -> FDI si tu ne l'as pas déjà
idx_to_fdi = {idx: fdi for fdi, idx in fdi_to_idx.items()}
num_classes = len(idx_to_fdi)

correct_per_class = torch.zeros(num_classes, dtype=torch.long)
total_per_class   = torch.zeros(num_classes, dtype=torch.long)

model.eval()
with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        # pts : (B, N, 3), lbls_fdi : (B, N) avec FDI
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)              # (B, N, C)
        preds_idx = logits.argmax(dim=-1)  # (B, N)

        for c in range(num_classes):
            mask = (lbls_idx == c)
            if mask.any():
                total_per_class[c]   += mask.sum().item()
                correct_per_class[c] += (preds_idx[mask] == c).sum().item()

# Affichage des accuracies par FDI
for c in range(num_classes):
    if total_per_class[c] == 0:
        continue
    fdi = idx_to_fdi[c]
    acc = correct_per_class[c].item() / total_per_class[c].item()
    print(f"FDI {fdi}: acc = {acc:.3f} (N = {total_per_class[c].item()})")

FDI 11: acc = 0.998 (N = 47196)
FDI 12: acc = 0.983 (N = 33572)
FDI 13: acc = 0.983 (N = 32292)
FDI 14: acc = 0.993 (N = 42129)
FDI 15: acc = 0.913 (N = 44163)
FDI 16: acc = 0.939 (N = 71742)
FDI 17: acc = 0.912 (N = 36725)
FDI 18: acc = 0.998 (N = 1786)
FDI 21: acc = 0.987 (N = 48387)
FDI 22: acc = 0.947 (N = 34578)
FDI 23: acc = 0.927 (N = 33068)
FDI 24: acc = 0.894 (N = 42438)
FDI 25: acc = 0.803 (N = 44018)
FDI 26: acc = 0.863 (N = 71985)
FDI 27: acc = 0.876 (N = 38490)
FDI 28: acc = 0.997 (N = 1544)
FDI 31: acc = 0.972 (N = 36656)
FDI 32: acc = 0.979 (N = 39940)
FDI 33: acc = 0.969 (N = 44446)
FDI 34: acc = 0.955 (N = 49451)
FDI 35: acc = 0.879 (N = 57516)
FDI 36: acc = 0.893 (N = 94001)
FDI 37: acc = 0.860 (N = 53169)
FDI 38: acc = 0.987 (N = 2557)
FDI 41: acc = 0.952 (N = 36342)
FDI 42: acc = 0.934 (N = 39690)
FDI 43: acc = 0.907 (N = 45194)
FDI 44: acc = 0.926 (N = 50965)
FDI 45: acc = 0.844 (N = 56758)
FDI 46: acc = 0.860 (N = 91807)
FDI 47: acc = 0.915 (N = 49838)
FDI 48: acc

In [12]:
model.eval()
total_tooth = 0
correct_tooth = 0

with torch.no_grad():
    for pts, lbls_fdi in test_loader:
        pts = pts.to(device)
        lbls_idx = encode_labels_torch(lbls_fdi).to(device)  # (B, N)

        logits = model(pts)
        preds_idx = logits.argmax(dim=-1)                    # (B, N)

        # même masque que dans la boucle d'entraînement
        mask = (lbls_idx != IGNORE_INDEX)

        correct_tooth += (preds_idx[mask] == lbls_idx[mask]).sum().item()
        total_tooth += mask.sum().item()

tooth_acc = correct_tooth / total_tooth
print("Tooth-only accuracy (FDI != 0):", tooth_acc)



Tooth-only accuracy (FDI != 0): 0.9162317782985132


In [14]:
from pathlib import Path

SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

ckpt_path = SAVE_DIR / "best_DGCNNSeg4D_no_gencive.pt"

torch.save(best_state, ckpt_path)
print("Checkpoint sauvé dans :", ckpt_path)

Checkpoint sauvé dans : checkpoints\best_DGCNNSeg4D_no_gencive.pt


In [ ]:
import collections

train_counts = collections.Counter()
test_counts  = collections.Counter()

# train
for pts, lbls_fdi in train_loader:
    # lbls_fdi : (B, N) FDI
    vals, counts = torch.unique(lbls_fdi, return_counts=True)
    for v, c in zip(vals.tolist(), counts.tolist()):
        train_counts[int(v)] += int(c)

# test
for pts, lbls_fdi in test_loader:
    vals, counts = torch.unique(lbls_fdi, return_counts=True)
    for v, c in zip(vals.tolist(), counts.tolist()):
        test_counts[int(v)] += int(c)

print("Train counts (par FDI, en nombre de points) :")
for fdi in sorted(train_counts.keys()):
    print(f"  FDI {fdi}: {train_counts[fdi]} points")

print("\nTest counts (par FDI, en nombre de points) :")
for fdi in sorted(test_counts.keys()):
    print(f"  FDI {fdi}: {test_counts[fdi]} points")


Train counts (par FDI, en nombre de points) :
  FDI 0: 2180253 points
  FDI 11: 94857 points
  FDI 12: 66282 points
  FDI 13: 65099 points
  FDI 14: 84967 points
  FDI 15: 88995 points
  FDI 16: 143275 points
  FDI 17: 75458 points
  FDI 18: 4170 points
  FDI 21: 96663 points
  FDI 22: 68661 points
  FDI 23: 64835 points
  FDI 24: 84670 points
  FDI 25: 87812 points
  FDI 26: 142672 points
  FDI 27: 76473 points
  FDI 28: 3742 points
  FDI 31: 72938 points
  FDI 32: 79010 points
  FDI 33: 86729 points
  FDI 34: 97532 points
  FDI 35: 114027 points
  FDI 36: 183051 points
  FDI 37: 106611 points
  FDI 38: 5074 points
  FDI 41: 72617 points
  FDI 42: 77735 points
  FDI 43: 87611 points
  FDI 44: 99684 points
  FDI 45: 111210 points
  FDI 46: 181486 points
  FDI 47: 105637 points
  FDI 48: 5364 points

Test counts (par FDI, en nombre de points) :
  FDI 0: 1083687 points
  FDI 11: 46805 points
  FDI 12: 33420 points
  FDI 13: 31921 points
  FDI 14: 41781 points
  FDI 15: 44287 points
  FDI

In [ ]:
from pathlib import Path

SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

ckpt_path = SAVE_DIR / "ToothFDIClassifier.pt"

torch.save({
    "model_state": model.state_dict(),
    "fdi_to_idx": fdi_to_idx,
    "idx_to_fdi": idx_to_fdi,
}, ckpt_path)
print("Checkpoint sauvé dans :", ckpt_path)

In [42]:
import torch
from pathlib import Path

save_path = Path("__data__/arc_jaw_datasets.pt")

torch.save(
    {
        "train": train_dataset,
        "test":  test_dataset,
    },
    save_path
)


In [5]:
dsets = torch.load(Path("__data__/arc_jaw_datasets.pt"),weights_only=False)
train_dataset = dsets["train"]
test_dataset  = dsets["test"]
